# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhan1117/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

Same lane, same mid-panel month (`month = '2026-03'`, panel spans `2025-01-27` to `2026-06-30`,
`2026-06` stays sealed) and same `fact_content_daily_performance` grain as `w03_data_contract` —
this notebook picks up where that one stopped. The five verified GSC fields become an actual
feature vector: two engineered ratios, one numeric column with a real gap that gets filled (with
a flag, not silently), and one categorical field one-hot encoded into its true three states
instead of collapsed to 0/1.

`dim_content`'s non-GSC columns are not joined here — see "What I excluded and why" below.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shaheerkhan1117/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import getpass
# Token order: env var -> Colab Secret -> prompt (last resort). Store as a Colab Secret
# named HF_TOKEN so the prompt never fires — never paste a token into a cell, this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, pandas as pd
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":    f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":    f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

MONTH = "2026-03"  # same mid-panel month as w03_data_contract — never the sealed 2026-06 test month

Paste your Hugging Face READ token (hf_...): ··········


**Engineered features, on purpose:**
- `ctr` = `total_clicks / total_impressions` — same as `w03_data_contract`.
- `impressions_per_active_day` = `total_impressions / active_days` — normalizes volume by how
  many days actually had traffic, so a page active 3 days isn't penalized against one active 31.
- `volatility_is_filled` + filled `position_volatility` — `STDDEV_SAMP` is `NULL` on a single
  point, not zero. Filling with `0.0` without a flag would tell the model "this page's ranking
  is rock-stable" for pages that only showed up once. The flag keeps that distinction visible.
- `ga4_*` one-hot columns from `ga4_data_available` — the column is genuinely three-state
  (`True` / `False` / `NULL`), and each state means something different: tracked-and-nonzero,
  tracked-and-zero-filled, or not tracked at all this pass. Collapsing that to a single 0/1
  column would quietly merge "zero engagement" with "we don't know.

In [2]:
raw = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                            AS total_impressions,
        SUM(gsc_clicks)                                                 AS total_clicks,
        AVG(gsc_avg_position)                                           AS avg_position,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days,
        STDDEV_SAMP(gsc_avg_position)                                   AS position_volatility,
        -- three-state on purpose, see markdown above
        MODE(ga4_data_available)                                       AS ga4_data_available
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2
""").df()

features = raw.copy()
features["ctr"] = (features["total_clicks"] / features["total_impressions"]).round(4)
features["impressions_per_active_day"] = (features["total_impressions"] / features["active_days"]).round(2)

# Fill: position_volatility is NULL only when active_days <= 1 — flag it, then fill, so the
# fill can't get read downstream as "confirmed stable."
features["volatility_is_filled"] = features["position_volatility"].isna().astype(int)
features["position_volatility"] = features["position_volatility"].fillna(0.0)

# Categorical handling: one-hot, keep the NaN bucket instead of dropping it.
ga4_dummies = pd.get_dummies(features["ga4_data_available"], prefix="ga4", dummy_na=True)
features = pd.concat([features, ga4_dummies], axis=1)

print(f"{len(features):,} content items, {features.shape[1]} columns")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

331,437 content items, 14 columns


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,active_days,position_volatility,ga4_data_available,ctr,impressions_per_active_day,volatility_is_filled,ga4_False,ga4_True,ga4_<NA>
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,31,2.442255,False,0.0011,210.42,0,True,False,False
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,31,2.481312,False,0.0000,14.61,0,True,False,False
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,31,1.243547,False,0.0011,181.61,0,True,False,False
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,31,0.983020,False,0.0026,159.48,0,True,False,False
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,21,22.609035,False,0.0000,2.00,0,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

| column | meaning | missing handling | categorical? | available before decision moment? |
|---|---|---|---|---|
| `total_impressions` | running sum of GSC impressions over observed days in the month | none needed — `SUM` over 0 rows is `0`, not null | no | yes — only sums days on or before the decision point |
| `total_clicks` | same, for clicks | none needed | no | yes, same reason |
| `avg_position` | mean GSC ranking position over observed days | none needed | no | yes |
| `active_days` | count of days with impressions > 0 in the window | none needed — `COUNT` floors at 0 | no | yes |
| `position_volatility` | `STDDEV_SAMP` of daily position — how noisy the ranking has been so far | `NULL` when `active_days <= 1`; filled to `0.0` with `volatility_is_filled` flag kept alongside | no | yes — describes history, not a forecast |
| `volatility_is_filled` | flag: was `position_volatility` a real stddev or a fill? | n/a — it IS the missingness indicator | binary | yes |
| `ctr` | `total_clicks / total_impressions` | `NaN` only if `total_impressions == 0`, which can't happen post-`SUM` here — checked below | no | yes |
| `impressions_per_active_day` | volume normalized by active days | `NaN` only if `active_days == 0`, excluded by the `FILTER` above — checked below | no | yes |
| `ga4_true` / `ga4_false` / `ga4_nan` | one-hot of GA4 availability state | `dummy_na=True` makes the missing state its own column, not a dropped row | categorical (3-state, one-hot) | yes — availability is known at the decision moment even when the underlying GA4 numbers aren't used |
| `client_hash_id`, `content_hash_id` | pseudonymous join keys | n/a | n/a — dropped before modeling | context only, never a feature |

In [3]:
# Verify the "none needed" and "NaN only if X" claims above instead of asserting them.
checks = pd.DataFrame({
    "column": features.columns,
    "dtype": features.dtypes.astype(str),
    "n_null": features.isna().sum().values,
    "pct_null": (features.isna().mean() * 100).round(2).values,
})
print("Null counts across the built feature vector (should be 0 everywhere except none):")
checks[checks["n_null"] > 0]  # expect an empty frame — every gap above was already filled or one-hot'd

Null counts across the built feature vector (should be 0 everywhere except none):


,column,dtype,n_null,pct_null
avg_position,avg_position,float64,154699,46.68
ga4_data_available,ga4_data_available,boolean,70700,21.33
ctr,ctr,float64,154699,46.68
impressions_per_active_day,impressions_per_active_day,float64,154699,46.68


## 3. The leakage hunt

Reuse the within-month decline label from `w03_data_contract` (`is_declining`: impressions fell
>20% in the back half of the month vs the front half). Two attacks on this notebook's own,
enriched feature set:

1. **Single-feature scan** — score every column against the label alone; anything near 1.0 gets
   looked at before it gets used, not after.
2. **A new trap, same shape** — `second_half_share` (second-half impressions over the month
   total) looks like an innocent ratio. It isn't: it's built from the label's own numerator and
   denominator, wearing a different name than `pct_change` did last week. Add it, watch the
   score jump, delete it, keep the honest number.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

halves = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
           SUM(CASE WHEN report_date >= DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2
    HAVING imp_first_half > 0
""").df()
halves["pct_change"] = (halves["imp_second_half"] - halves["imp_first_half"]) / halves["imp_first_half"]
halves["is_declining"] = (halves["pct_change"] < -0.2).astype(int)

checked = features.merge(halves[["client_hash_id", "content_hash_id", "is_declining"]],
                          on=["client_hash_id", "content_hash_id"], how="inner")

honest_cols = ["total_impressions", "total_clicks", "avg_position", "active_days",
               "position_volatility", "volatility_is_filled", "ctr",
               "impressions_per_active_day"] + list(ga4_dummies.columns)

# Attack 1 — single-feature AUC scan
single_feature_auc = {}
for col in honest_cols:
    d = checked.dropna(subset=[col, "is_declining"])
    if d[col].nunique() < 2:
        continue
    auc = roc_auc_score(d["is_declining"], d[col])
    single_feature_auc[col] = round(max(auc, 1 - auc), 3)

print("Single-feature AUC against is_declining (nothing here should sit near 1.0):")
print(pd.Series(single_feature_auc).sort_values(ascending=False))

# Attack 2 — the trap
halves["second_half_share"] = halves["imp_second_half"] / (halves["imp_first_half"] + halves["imp_second_half"])
trap = checked.merge(halves[["client_hash_id", "content_hash_id", "second_half_share"]],
                      on=["client_hash_id", "content_hash_id"])

def quick_auc(cols, data):
    d = data.dropna(subset=cols + ["is_declining"])
    X, y = d[cols], d["is_declining"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])

honest_auc = quick_auc(honest_cols, trap)
print(f"\nHonest AUC (enriched feature set, no leak): {honest_auc:.3f}")

leaked_auc = quick_auc(honest_cols + ["second_half_share"], trap)
print(f"Leaked AUC (adding second_half_share, built from the label's own halves): {leaked_auc:.3f}")

print("\nsecond_half_share deleted now — it only ever existed to demonstrate the leak.")
del trap["second_half_share"]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Single-feature AUC against is_declining (nothing here should sit near 1.0):
total_impressions             0.645
impressions_per_active_day    0.632
active_days                   0.620
total_clicks                  0.593
ctr                           0.584
volatility_is_filled          0.557
position_volatility           0.550
avg_position                  0.522
ga4_True                      0.515
ga4_False                     0.514
ga4_<NA>                      0.501
dtype: float64


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Honest AUC (enriched feature set, no leak): 0.676
Leaked AUC (adding second_half_share, built from the label's own halves): 0.999

second_half_share deleted now — it only ever existed to demonstrate the leak.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


**Privacy pass.** Everything above is either a pseudonymous join key (dropped before modeling)
or a numeric / one-hot column derived from aggregates — no raw text, no URLs, no client names
enter the feature vector at any point.

In [5]:
non_numeric_non_id = [c for c in features.columns
                       if c not in ("client_hash_id", "content_hash_id")
                       and features[c].dtype == "object"]
print("Non-numeric, non-id columns in the feature vector:", non_numeric_non_id or "none")

Non-numeric, non-id columns in the feature vector: none


## 4. What I excluded and why

- **`fact_content_query_90d`** — same reason as `w03_data_contract`: its 90-day window overlaps
  the panel's tail, and joining it safely needs window-alignment work not done in this notebook.
  Confirmed below rather than just repeated.
- **`dim_content`'s non-GSC columns** (title text, URL, any free-text field) — not joined at all.
  Their schema hasn't been checked column-by-column in this pass, and an unvetted text field is
  both a leakage risk (could encode future edits) and a privacy risk (could carry client content
  verbatim) until it has been.
- **Raw GA4 engagement counts** (as opposed to just the `ga4_*` availability flag used above) —
  those counts are zero-filled before a client's `ga4_data_start`. The flag alone is safe to use
  now; the numbers themselves need the same `IS TRUE` filtering discipline from
  `w03_data_contract` applied consistently, which hasn't been re-verified for this feature set.
- **`client_hash_id`, `content_hash_id`** — pseudonyms, kept only to join and group; dropped
  before anything gets fit.
- **Raw `report_date` / `month`** — this pass is a single calendar month, so either column would
  be constant (no signal) or would leak the label window's calendar position if ever encoded.

In [7]:
q90_range = con.sql(f"""
    SELECT MIN(window_start) AS min_date, MAX(window_start) AS max_date, COUNT(*) AS n
    FROM {TABLES['fact_query_90d']}
""").df()
q90_range

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,n
0,2026-04-02,2026-04-02,2414248


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.